### Question

1. What the value of the portfolio will be at a given entry level, buying regularly at a given annual return over a given period of time?

### Import libraries

The libraries set up a basic Dash web application that uses Plotly and pandas to create interactive visualizations based on user input.

In [1]:
import dash
from dash import dcc, html, Input, Output
import plotly.graph_objs as go
import pandas as pd

These style dictionaries define CSS properties in Python for styling HTML elements in a Dash app—specifically for making labels bold, styling input fields with padding and full width, and adding padding to div containers.

In [2]:
label_style = {
    'fontWeight': 'bold'
}

input_style = {
    'width': '100%',
    'padding': '6px',
    'marginBottom': '8px',
    'fontWeight': 'bold'
}

div_style = {
    'padding': '0px 0px 5px'
}

This part of the script defines the layout of the Dash web app, creating a user interface for an "Investment Growth Calculator". It includes a header and two main sections: a left panel with input fields for financial parameters (initial capital, monthly investment, annual return, investment period, and y-axis scale), and a right panel for displaying a graph. The layout uses html.Div elements styled with CSS for a responsive, visually appealing design using Flexbox. Overall, it sets up the structure and styling for user input and the output graph in the app interface.

In [3]:
app = dash.Dash(__name__)

app.layout = html.Div([
    html.H1("Investment Growth Calculator", style={
    'textAlign': 'center',
    'fontSize': '36px'
    }),

    html.Div([
        html.Div([
            html.Div([
                html.Label("Initial Capital", style=label_style),
                dcc.Input(id='initial-capital', type='number', value=10000, style=input_style),
        ], style = div_style),

            html.Div([
                html.Br(), html.Label("Monthly Investment", style=label_style),
                dcc.Input(id='monthly-investment', type='number', value=500, style=input_style),
        ], style = div_style),

            html.Div([
                html.Br(), html.Label("Annual Return (%)", style=label_style),
                dcc.Input(id='annual-return', type='number', value=7, style=input_style),
        ], style = div_style),

            html.Div([
                html.Br(), html.Label("Investment Period (years)", style=label_style),
                dcc.Input(id='investment-period', type='number', value=10, style=input_style),
        ], style = div_style),

            html.Div([
                html.Br(), html.Label("Scale of Y-axis", style=label_style),
                dcc.RadioItems(id='scale-selector',
                    options=[
                        {'label': 'Linear', 'value': 'linear'},
                        {'label': 'Logarithmic', 'value': 'log'}
                    ],
                    value='linear',
                    labelStyle={'display': 'inline-block','padding': '6px','marginTop': '6px','marginRight': '10px'})
        ], style=div_style),

        ], style={
            'flex': '1',
            'padding': '10px',            
            'display': 'flex',
            'flexDirection': 'column',
            'justifyContent': 'space-evenly',
            'minHeight': '500px',
            'minWidth': '250px'
            }
        ),

        html.Div([
            dcc.Graph(id='investment-graph')
        ], style={'flex': '2', 'padding': '20px', 'margin':'10px'}),
    ], style={'display': 'flex', 'flexDirection': 'row'})
], style={'font-family': 'Verdana', "backgroundColor": "rgb(230, 230, 0)", 'color': '#1a1aff', 'padding': '10px'})

This part of code defines the interactive behavior of the Dash app using a callback function that updates the investment growth graph whenever any input value changes. It calculates the monthly investment growth over the selected time period, accounting for compound interest, and splits the results into invested capital, gains, and total value. The data is stored in a DataFrame and visualized using a Plotly line chart with three traces representing principal, gains, and total value. Finally, it customizes the layout of the graph and runs the Dash app if the script is executed directly.

In [4]:
@app.callback(
    Output('investment-graph', 'figure'),
    Input('initial-capital', 'value'),
    Input('monthly-investment', 'value'),
    Input('annual-return', 'value'),
    Input('investment-period', 'value'),
    Input('scale-selector', 'value')
)
def update_graph(initial_capital, monthly_investment, annual_return, years, scale_type):
    months = years * 12
    monthly_return = (1 + annual_return / 100) ** (1 / 12) - 1

    total_value = []
    principal = []
    gains = []

    value = initial_capital
    invested = initial_capital

    for month in range(months + 1):
        total_value.append(value)
        principal.append(invested)
        gains.append(value - invested)

        value = value * (1 + monthly_return) + monthly_investment
        invested += monthly_investment

    df = pd.DataFrame({
        'Month': list(range(months + 1)),
        'Year': [round(m / 12, 2) for m in range(months + 1)],
        'Total Value': total_value,
        'Principal': principal,
        'Gains': gains
    })

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=df['Year'], y=df['Principal'],
                             mode='lines', name='Invested Capital'))
    fig.add_trace(go.Scatter(x=df['Year'], y=df['Gains'],
                             mode='lines', name='Interest Gains'))
    fig.add_trace(go.Scatter(x=df['Year'], y=df['Total Value'],
                             mode='lines', name='Total Value', line=dict(dash='dash')))

    fig.update_layout(title={
                        'text': "<b>Investment Growth Over Time</b>",
                        'x': 0.5,
                        'xanchor': 'center',
                        'font': dict(
                            size=24,
                            color='#1a1aff')
                    },
                    xaxis_title='Year',
                    yaxis_title='Value (CZK)',
                    yaxis_type=scale_type,
                    font=dict(family="Verdana", size=14, color="#1a1a1a"),
                    plot_bgcolor="#f0f0f0",
                    paper_bgcolor="#fa9c1c",
                    legend=dict(orientation="h", font=dict(color="#1a1a1a"), x=0, y=1.15),
                    hovermode='x unified')
    return fig

if __name__ == '__main__':
    app.run(debug=True)
